# LIBRARY

In [39]:
import json
import re

REQUIRED_FIELDS = ["time", "src", "dst", "protocol", "length", "info"]
VALID_PROTOCOLS = ["ZigBee", "ZigBee HA"]


def validate_packet_dict(pkt):

    # Required fields
    for f in REQUIRED_FIELDS:
        if f not in pkt:
            return False, f"Missing field: {f}", pkt

    # Unexpected fields
    extras = [k for k in pkt.keys() if k not in REQUIRED_FIELDS]
    if extras:
        return False, f"Unexpected fields: {extras}", pkt

    # Validate time
    try:
        float(pkt["time"])
    except:
        return False, "Time not numeric", pkt

    # Validate length
    try:
        int(pkt["length"])
    except:
        return False, "Length not numeric", pkt

    # Validate protocol
    if pkt["protocol"] not in VALID_PROTOCOLS:
        return False, f"Invalid protocol: {pkt['protocol']}", pkt

    return True, None, pkt


def extract_json(raw_text):
    """Extract JSON from GPT output using multiple fallbacks."""

    # 1) Try direct JSON
    try:
        obj = json.loads(raw_text)
        if isinstance(obj, list):
            return obj
    except:
        pass

    # 2) Try inside ```json ... ```
    block = re.search(r"```json(.*?)```", raw_text, re.DOTALL)
    if block:
        try:
            obj = json.loads(block.group(1).strip())
            return obj
        except:
            pass

    # 3) Try any [ ... ] array
    array_match = re.search(r"(\[.*\])", raw_text, re.DOTALL)
    if array_match:
        try:
            obj = json.loads(array_match.group(1).strip())
            return obj
        except:
            pass

    # Fail
    return None


def analyze_llm_output(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        raw = f.read()

    packets = extract_json(raw)

    if packets is None:
        print("❌ ERROR: Could not decode JSON.")
        return [], [raw], 0.0

    if not isinstance(packets, list):
        print("❌ ERROR: JSON is not a list of packets.")
        return [], [], 0.0

    valid_packets = []
    corrupt_packets = []

    last_time = -1
    timestamp_errors = 0

    for i, pkt in enumerate(packets):

        is_valid, error, pkt = validate_packet_dict(pkt)

        if is_valid:
            # Check timestamp ordering
            t = float(pkt["time"])
            if t < last_time:
                timestamp_errors += 1
            last_time = t

            valid_packets.append(pkt)

        else:
            corrupt_packets.append((i, pkt, error))

    total = len(packets)
    corrupt_ratio = (len(corrupt_packets) / max(total, 1)) *100 
    decodability = 100 - corrupt_ratio

    print("\n===== PACKET QUALITY REPORT =====")
    print(f"Total packets: {total}")
    print(f"Valid packets: {len(valid_packets)}")
    print(f"Corrupt packets: {len(corrupt_packets)}")
    print(f"Timestamp ordering issues: {timestamp_errors}")
    print(f"Decodability: {decodability:.2f}%")
    print("=================================\n")

    return valid_packets, corrupt_packets, decodability


In [40]:
import json
import re

REQUIRED_FIELDS = ["time", "src", "dst", "protocol", "length", "info"]
VALID_PROTOCOLS = ["ZigBee", "ZigBee HA"]

VALID_SOURCES = ["0x1de6"] # only for experiment #1
VALID_DESTINATIONS = ["0xd7a7", "0xfffc"] # only for experiment #1

def check_protocol_compliance(valid_packets):
    """
    Checks whether each packet uses the correct protocol for its message type.
    Rules:
        - If 'Link Status' → protocol must be 'ZigBee'
        - Otherwise        → protocol must be 'ZigBee HA'
    """

    incorrect = []

    for i, pkt in enumerate(valid_packets):
        info = pkt["info"]
        protocol = pkt["protocol"]

        if "Link Status" in info:
            expected = "ZigBee"
        else:
            expected = "ZigBee HA"

        if protocol != expected:
            incorrect.append((i, pkt, expected))

    total = len(valid_packets)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total > 0 else 0

    print("\n===== PROTOCOL COMPLIANCE REPORT =====")
    print(f"Total packets: {total}")
    print(f"Incorrect protocol packets: {error_count}")
    print(f"Protocol Compliance: {compliance:.2f}%")
    print("=============================================\n")

    if incorrect:
        print("---- Incorrect Protocol Packets ----")
        for idx, pkt, expected in incorrect[:20]:
            print(f"[Packet {idx}] Expected: {expected}, Found: {pkt['protocol']}")
            print(f"   Info: {pkt['info']}")
            print(f"   Packet: {pkt}\n")

    return compliance, incorrect

def check_address_compliance(valid_packets):
    """
    Checks address correctness rules for ZigBee:
        - Link Status → dst must be 0xfffc
        - src ≠ dst
    """

    incorrect = []

    for i, pkt in enumerate(valid_packets):
        src = pkt["src"]
        dst = pkt["dst"]
        info = pkt["info"]

        # Rule 1: Link Status must be broadcast
        if "Link Status" in info and dst != "0xfffc":
            incorrect.append((i, pkt, "dst must be 0xfffc for Link Status"))

        # Rule 2: src != dst
        if src == dst:
            incorrect.append((i, pkt, "src and dst cannot be identical"))

         # --- Rule 3: Invalid source address ---
        if src not in VALID_SOURCES:
            incorrect.append((i, pkt, f"Invalid src: {src} (must be one of {VALID_SOURCES})"))

         # --- Rule 4: Invalid destination address ---
        if dst not in VALID_DESTINATIONS:
            incorrect.append((i, pkt, f"Invalid dst: {dst} (must be one of {VALID_DESTINATIONS})"))


    total = len(valid_packets)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total > 0 else 0

    print("\n===== ADDRESS COMPLIANCE REPORT =====")
    print(f"Total packets: {total}")
    print(f"Packets violating address rules: {error_count}")
    print(f"Address Compliance: {compliance:.2f}%")
    print("===========================================\n")

    if incorrect:
        print("---- Incorrect Address Packets ----")
        for idx, pkt, reason in incorrect[:20]:
            print(f"[Packet {idx}] Error: {reason}")
            print(f"   Packet: {pkt}\n")

    return compliance, incorrect

In [41]:
import re

def check_seq_ordering(valid_packets, max_step=50):
    """
    Checks ZCL sequence correctness with wrap-around logic (0–255).
    
    Rules:
        1. Sequence number must be in range [0, 255]
        2. Sequence ordering must increase modulo 256
           → 250 → 5 is allowed
           → 250 → 200 is NOT allowed
           → Large backward jumps (e.g., 180 → 20) are NOT allowed
    """

    # Sort by timestamp
    sorted_packets = sorted(valid_packets, key=lambda p: float(p["time"]))

    seq_packets = []
    seq_values = []

    # Extract ZCL sequences
    for pkt in sorted_packets:
        info = pkt["info"]

        if "ZCL" in info and "Seq:" in info:
            match = re.search(r"Seq:\s*(\d+)", info)
            if match:
                seq = int(match.group(1))

                # Rule 1 — sequence must be in allowed range
                if seq < 0 or seq > 255:
                    seq_packets.append(pkt)
                    seq_values.append(seq)
                    continue

                seq_packets.append(pkt)
                seq_values.append(seq)

    incorrect = []

    # Rule 2 — valid monotonicity with wrap-around
    for i in range(1, len(seq_values)):
        prev = seq_values[i - 1]
        curr = seq_values[i]

        # Compute modular difference
        diff = (curr - prev) % 256

        # diff must be in [1, max_step]
        if diff == 0 or diff > max_step:
            incorrect.append((i, seq_packets[i], prev, curr, diff))

    total = len(seq_values)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total > 0 else 0

    print("\n===== ZCL SEQUENCE ORDER REPORT =====")
    print(f"Total ZCL packets: {total}")
    print(f"Out-of-order or invalid jumps: {error_count}")
    print(f"Sequence Compliance: {compliance:.2f}%")
    print("=====================================\n")

    if incorrect:
        print("---- Incorrect SEQ Packets ----")
        for idx, pkt, prev, curr, diff in incorrect[:20]:
            print(f"[Packet index {idx}] Prev={prev}, Curr={curr}, Jump={diff}")
            print(f"   Time: {pkt['time']}")
            print(f"   Info: {pkt['info']}\n")

    return compliance, incorrect

In [42]:
# from collections import Counter

# def check_repetition_rate(valid_packets, top_n=10):
#     """
#     Computes repetition only for NON-Link-Status ZigBee packets.
#     Groups packets by (src, dst, protocol, length, info).
#     """

#     keys = []

#     for pkt in valid_packets:
#         info = pkt["info"]

#         # Skip Link Status packets
#         if "Link Status" in info:
#             continue

#         key = (pkt["src"], pkt["dst"], pkt["protocol"], pkt["length"], pkt["info"])
#         keys.append(key)

#     counter = Counter(keys)
#     total_packets = len(keys)  # Only non-Link-Status packets

#     repeated_packets = sum(count - 1 for count in counter.values() if count > 1)

#     repetition_rate = (repeated_packets / max(1, total_packets)) * 100

#     print("\n===== PACKET REPETITION REPORT (NO LINK-STATUS) =====")
#     print(f"Total NON-Link-Status packets: {total_packets}")
#     print(f"Repeated packets: {repeated_packets}")
#     print(f"Repetition rate: {repetition_rate:.2f}%")
#     print("======================================================\n")

#     print(f"---- TOP {top_n} MOST REPEATED PATTERNS ----")
#     for key, count in counter.most_common(top_n):
#         if count > 1:
#             src, dst, protocol, length, info = key
#             print(f"{count}× | {src} → {dst} | {protocol} | len={length} | {info}")
#         else:
#             break

#     return repetition_rate, counter, repeated_packets


from collections import Counter

def check_repetition_rate(valid_packets, top_n=10):
    """
    Computes repetition only for NON-Link-Status ZigBee packets.
    Groups packets by (src, dst, protocol, length, info).
    """

    keys = []

    for pkt in valid_packets:
        info = pkt["info"]

        # Skip Link Status packets
        if "Link Status" in info:
            continue

        key = (pkt["src"], pkt["dst"], pkt["protocol"], pkt["length"], pkt["info"])
        keys.append(key)

    counter = Counter(keys)
    total_packets = len(keys)  # Only non-Link-Status packets

    repeated_packets = sum(count - 1 for count in counter.values() if count > 1)

    repetition_rate = (repeated_packets / max(1, total_packets)) * 100

    print("\n===== PACKET REPETITION REPORT (NO LINK-STATUS) =====")
    print(f"Total NON-Link-Status packets: {total_packets}")
    print(f"Repeated packets: {repeated_packets}")
    print(f"Repetition rate: {repetition_rate:.2f}%")
    print("======================================================\n")

    print(f"---- TOP {top_n} MOST REPEATED PATTERNS ----")
    for key, count in counter.most_common(top_n):
        if count > 1:
            src, dst, protocol, length, info = key
            print(f"{count}× | {src} → {dst} | {protocol} | len={length} | {info}")
        else:
            break

    return repetition_rate, counter, repeated_packets

In [43]:
def compute_exact_match_rate(valid_packets, sample_packets):
    """
    Computes Exact Match Rate between generated packets and real sample packets.
    Comparison rules:
        - time: only integer part must match
        - src, dst, protocol, length, info: must match exactly
    """

    def normalize(pkt):
        """Convert packet to a comparison key with integer time."""
        return (
            int(float(pkt["time"])),   # integer time only
            pkt["src"],
            pkt["dst"],
            pkt["protocol"],
            pkt["length"],
            pkt["info"]
        )

    # Convert real sample packets into a set for fast lookup
    sample_set = {normalize(pkt) for pkt in sample_packets}

    exact_matches = []
    total_gen = len(valid_packets)

    for pkt in valid_packets:
        key = normalize(pkt)
        if key in sample_set:
            exact_matches.append(pkt)

    exact_count = len(exact_matches)
    exact_match_rate = (exact_count / max(1, total_gen)) * 100

    print("\n===== EXACT MATCH RATE REPORT =====")
    print(f"Total generated packets: {total_gen}")
    print(f"Exact matches with real data: {exact_count}")
    print(f"Exact Match Rate: {exact_match_rate:.2f}%")
    print("===================================\n")

    if exact_matches:
        print("---- Example Exact-Matched Packets (first 10) ----")
        for pkt in exact_matches[:10]:
            print(pkt)
            print()

    return exact_match_rate, exact_matches

In [44]:

# from collections import Counter
# def check_repetition_rate(valid_packets, top_n=10):
#     """
#     Groups packets by (src, dst, protocol, info) and computes:
#       - count of each combination
#       - repetition percentage
#     """

#     # Build tuple key for each packet
#     keys = []
#     for pkt in valid_packets:
#         key = (pkt["src"], pkt["dst"], pkt["protocol"], pkt["info"])
#         keys.append(key)

#     counter = Counter(keys)
#     total_packets = len(valid_packets)

#     # repetition = total occurrences minus unique patterns
#     repeated_packets = sum(count - 1 for count in counter.values() if count > 1)

#     repetition_rate = (repeated_packets / max(1, total_packets)) * 100

#     print("\n===== PACKET REPETITION REPORT =====")
#     print(f"Total packets: {total_packets}")
#     print(f"Repeated packets: {repeated_packets}")
#     print(f"Repetition rate: {repetition_rate:.2f}%")
#     print("=====================================\n")

#     # Show most repeated patterns
#     print(f"---- TOP {top_n} MOST FREQUENT PATTERNS ----")
#     for (src, dst, protocol, info), count in counter.most_common(top_n):
#         if count > 1:
#             print(f"{count}×  |  {src} → {dst} | {protocol} | {info}")
#         else:
#             break

#     return repetition_rate, counter,repeated_packets 

# RESULTS

In [45]:
import json
import os

data = []
with open('../Datasets/Experiment_1_one_way_communication_10_minute_input_sample.json', 'r') as file:
    for line in file:
        data.append(json.loads(line))
""" Change Broadcast with the address """
for item in data:
    if item['Destination'] == 'Broadcast':
        item['Destination'] = '0xfffc'
""" Create sample prompt from json file """
Sample_Packets= []
for item in data:
    sample_packets = {
        'time': str(item['Time']),
        'src': item['Source'],
        'dst': item['Destination'],
        'protocol': item['Protocol'],
        'length': str(item['Length']),
        'info': item['Info']
    }
    Sample_Packets.append(sample_packets)

# print(Sample_Packets)

## EXPERIMENT 1

In [46]:
print("\n===== SUMMARY REPORT GPT 5 - LOW  =====")

import pandas as pd
Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(1,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/New_Experiments/GPT5/GPT5_Exp1_Reasoning_low_Trial_{n}_10_minute_generated_message_second.json"

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
    address_compliance, address_errors = check_address_compliance(valid_packets)
    seq_compliance, seq_errors = check_seq_ordering(valid_packets)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
    emr, matched_list = compute_exact_match_rate(valid_packets, Sample_Packets)



    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average EMR Rate: {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT GPT 5 - LOW  =====

===== PACKET QUALITY REPORT =====
Total packets: 196
Valid packets: 196
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PROTOCOL COMPLIANCE REPORT =====
Total packets: 196
Incorrect protocol packets: 0
Protocol Compliance: 100.00%


===== ADDRESS COMPLIANCE REPORT =====
Total packets: 196
Packets violating address rules: 0
Address Compliance: 100.00%


===== ZCL SEQUENCE ORDER REPORT =====
Total ZCL packets: 156
Out-of-order or invalid jumps: 0
Sequence Compliance: 100.00%


===== PACKET REPETITION REPORT (NO LINK-STATUS) =====
Total NON-Link-Status packets: 156
Repeated packets: 0
Repetition rate: 0.00%

---- TOP 10 MOST REPEATED PATTERNS ----

===== EXACT MATCH RATE REPORT =====
Total generated packets: 196
Exact matches with real data: 2
Exact Match Rate: 1.02%

---- Example Exact-Matched Packets (first 10) ----
{'time': '0.0', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link

In [47]:
print("\n===== SUMMARY REPORT GPT 5 - MEDIUM  =====")

import pandas as pd
Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(1,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/New_Experiments/GPT5/GPT5_Exp1_Reasoning_medium_Trial_{n}_10_minute_generated_message.json"

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
    address_compliance, address_errors = check_address_compliance(valid_packets)
    seq_compliance, seq_errors = check_seq_ordering(valid_packets)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
    emr, matched_list = compute_exact_match_rate(valid_packets, Sample_Packets)



    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average EMR Rate: {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")



===== SUMMARY REPORT GPT 5 - MEDIUM  =====

===== PACKET QUALITY REPORT =====
Total packets: 188
Valid packets: 188
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PROTOCOL COMPLIANCE REPORT =====
Total packets: 188
Incorrect protocol packets: 0
Protocol Compliance: 100.00%


===== ADDRESS COMPLIANCE REPORT =====
Total packets: 188
Packets violating address rules: 0
Address Compliance: 100.00%


===== ZCL SEQUENCE ORDER REPORT =====
Total ZCL packets: 146
Out-of-order or invalid jumps: 0
Sequence Compliance: 100.00%


===== PACKET REPETITION REPORT (NO LINK-STATUS) =====
Total NON-Link-Status packets: 146
Repeated packets: 0
Repetition rate: 0.00%

---- TOP 10 MOST REPEATED PATTERNS ----

===== EXACT MATCH RATE REPORT =====
Total generated packets: 188
Exact matches with real data: 11
Exact Match Rate: 5.85%

---- Example Exact-Matched Packets (first 10) ----
{'time': '0.0', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': '

In [48]:
print("\n===== SUMMARY REPORT GPT 5 - HIGH  =====")

import pandas as pd
Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(1,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/New_Experiments/GPT5/GPT5_Exp1_Reasoning_high_Trial_{n}_10_minute_generated_message.json"

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
    address_compliance, address_errors = check_address_compliance(valid_packets)
    seq_compliance, seq_errors = check_seq_ordering(valid_packets)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
    emr, matched_list = compute_exact_match_rate(valid_packets, Sample_Packets)



    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average EMR Rate: {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")



===== SUMMARY REPORT GPT 5 - HIGH  =====

===== PACKET QUALITY REPORT =====
Total packets: 156
Valid packets: 156
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PROTOCOL COMPLIANCE REPORT =====
Total packets: 156
Incorrect protocol packets: 0
Protocol Compliance: 100.00%


===== ADDRESS COMPLIANCE REPORT =====
Total packets: 156
Packets violating address rules: 0
Address Compliance: 100.00%


===== ZCL SEQUENCE ORDER REPORT =====
Total ZCL packets: 116
Out-of-order or invalid jumps: 0
Sequence Compliance: 100.00%


===== PACKET REPETITION REPORT (NO LINK-STATUS) =====
Total NON-Link-Status packets: 116
Repeated packets: 0
Repetition rate: 0.00%

---- TOP 10 MOST REPEATED PATTERNS ----

===== EXACT MATCH RATE REPORT =====
Total generated packets: 156
Exact matches with real data: 3
Exact Match Rate: 1.92%

---- Example Exact-Matched Packets (first 10) ----
{'time': '0.0', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Lin

## EXPERIMENT 1 - 30 MINUTES

In [49]:
print("\n===== SUMMARY REPORT  =====")
import pandas as pd
Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []


filepath = f"../Generated_Traffic/New_Experiments/Generalization_Ability/GPT5_Exp1_30_minute_generated_message.json"

valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
address_compliance, address_errors = check_address_compliance(valid_packets)
seq_compliance, seq_errors = check_seq_ordering(valid_packets)
repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
emr, matched_list = compute_exact_match_rate(valid_packets, Sample_Packets)



Decodability.append(decodability)
Protocol_Compliance_Rate.append(protocol_compliance)
Address_Compliance_Rate.append(address_compliance)
Seq_Compliance_Rate.append(seq_compliance)
Repetition_Rate.append(repetition_rate)
Exact_Match_Rate.append(emr)


print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average EMR Rate: {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT  =====

===== PACKET QUALITY REPORT =====
Total packets: 264
Valid packets: 264
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PROTOCOL COMPLIANCE REPORT =====
Total packets: 264
Incorrect protocol packets: 0
Protocol Compliance: 100.00%


===== ADDRESS COMPLIANCE REPORT =====
Total packets: 264
Packets violating address rules: 0
Address Compliance: 100.00%


===== ZCL SEQUENCE ORDER REPORT =====
Total ZCL packets: 145
Out-of-order or invalid jumps: 0
Sequence Compliance: 100.00%


===== PACKET REPETITION REPORT (NO LINK-STATUS) =====
Total NON-Link-Status packets: 145
Repeated packets: 0
Repetition rate: 0.00%

---- TOP 10 MOST REPEATED PATTERNS ----

===== EXACT MATCH RATE REPORT =====
Total generated packets: 264
Exact matches with real data: 3
Exact Match Rate: 1.14%

---- Example Exact-Matched Packets (first 10) ----
{'time': '0.0', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}

{

# EXPERIMENT 2

In [50]:

REQUIRED_FIELDS = ["time", "src", "dst", "protocol", "length", "info"]
VALID_PROTOCOLS = ["ZigBee", "ZigBee HA"]

VALID_SOURCES = ["0x1de6", "0xd7a7"] # only for experiment #1
VALID_DESTINATIONS = ["0x1de6", "0xd7a7", "0xfffc"]# only for experiment #1

In [51]:
import json
import os

data = []

with open(r'../Datasets/Experiment_2_input_sample.json', 'r') as file:
    for line in file:
        data.append(json.loads(line))
""" Change Broadcast with the address """
for item in data:
    if item['Destination'] == 'Broadcast':
        item['Destination'] = '0xfffc'
""" Create sample packets from json file """
Sample_Packets = []
for item in data:
    sample_packets = {
        'time': str(item['Time']),
        'src': item['Source'],
        'dst': item['Destination'],
        'protocol': item['Protocol'],
        'length': str(item['Length']),
        'info': item['Info']
    }
    Sample_Packets.append(sample_packets)
    #print(sample_prompt)


In [52]:
import re

def check_address_compliance_exp2(valid_packets, VALID_SOURCES, VALID_DESTINATIONS):
    """
    Experiment 2 — Address Compliance + Request/Response correctness.
    Differences vs Experiment 1:
        - Response with no request is NOT an error.
        - Request with no response is NOT an error.
        - Only if both exist (same seq), check direction + src!=dst.
    """

    incorrect = []

    # Sort packets by timestamp
    packets_sorted = sorted(valid_packets, key=lambda p: float(p["time"]))

    # Regex for sequence number
    seq_pattern = re.compile(r"Seq:\s*(\d+)")

    # Dictionaries to store requests and responses
    req_dict = {}   # seq → request packet
    res_dict = {}   # seq → response packet

    # ===========================================================
    # PASS 1 — BASIC ADDRESS RULES (apply to EVERY PACKET)
    # ===========================================================
    for i, pkt in enumerate(packets_sorted):

        src = pkt["src"]
        dst = pkt["dst"]
        info = pkt["info"]

        # --- Rule A: Link Status must be broadcast ---
        if "Link Status" in info and dst != "0xfffc":
            incorrect.append((i, pkt, "Link Status must have dst = 0xfffc"))

        # --- Rule B: src != dst ---
        if src == dst:
            incorrect.append((i, pkt, "src and dst cannot be identical"))

        # --- Rule C: src must be in valid devices ---
        if src not in VALID_SOURCES:
            incorrect.append((i, pkt, f"Invalid src {src}, must be in {VALID_SOURCES}"))

        # --- Rule D: dst must be in valid devices ---
        if dst not in VALID_DESTINATIONS:
            incorrect.append((i, pkt, f"Invalid dst {dst}, must be in {VALID_DESTINATIONS}"))

        # Extract seq number if exists
        match = seq_pattern.search(info)
        if not match:
            continue

        seq = int(match.group(1))

        # Store request or response for later pair-check
        if "ZCL: Read Attributes," in info and "Response" not in info:
            req_dict[seq] = pkt

        elif "ZCL: Read Attributes Response" in info:
            res_dict[seq] = pkt

    # ===========================================================
    # PASS 2 — REQUEST / RESPONSE PAIRS (ONLY if both exist)
    # ===========================================================
    for seq, req_pkt in req_dict.items():

        # If matching response does not exist → ignore (not an error)
        if seq not in res_dict:
            continue

        res_pkt = res_dict[seq]
        req_src = req_pkt["src"]
        req_dst = req_pkt["dst"]

        # --- Rule E1: Response direction must be inverted ---
        if not (res_pkt["src"] == req_dst and res_pkt["dst"] == req_src):
            incorrect.append((
                -1,
                res_pkt,
                f"Seq {seq}: Response direction incorrect. Expected src={req_dst}, dst={req_src}"
            ))

        # --- Rule E2: Response must not have src == dst ---
        if res_pkt["src"] == res_pkt["dst"]:
            incorrect.append((
                -1,
                res_pkt,
                f"Seq {seq}: Response has identical src and dst"
            ))

    # ===========================================================
    # SUMMARY
    # ===========================================================    
    total = len(valid_packets)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total else 0

    print("\n===== ADDRESS COMPLIANCE — EXPERIMENT 2 =====")
    print(f"Total packets: {total}")
    print(f"Violations: {error_count}")
    print(f"Compliance: {compliance:.2f}%")
    print("=============================================\n")

    if incorrect:
        print("---- First Incorrect Packets ----")
        for idx, pkt, reason in incorrect[:20]:
            print(f"[Packet {idx}] Error: {reason}")
            print(pkt)
            print()

    return compliance, incorrect


In [53]:
import re

def check_seq_ordering_exp2(valid_packets, max_step=50):

    sorted_packets = sorted(valid_packets, key=lambda p: float(p["time"]))
    seq_pattern = re.compile(r"Seq:\s*(\d+)")

    req_list = []   # list of request SEQ (in time order)
    res_list = []   # list of response SEQ (in time order)

    req_seq_packets = []  # for reporting errors
    res_seq_packets = []

    for pkt in sorted_packets:
        info = pkt["info"]

        if "Seq:" not in info:
            continue

        match = seq_pattern.search(info)
        if not match:
            continue

        seq = int(match.group(1))

        # Rule 1: SEQ must be in valid range
        if seq < 0 or seq > 255:
            incorrect.append((pkt, f"SEQ out of range: {seq}"))
            continue

        # Classify request vs response
        if "ZCL: Read Attributes," in info and "Response" not in info:
            req_list.append(seq)
            req_seq_packets.append(pkt)

        elif "ZCL: Read Attributes Response" in info:
            res_list.append(seq)
            res_seq_packets.append(pkt)

    incorrect = []

    # ------------------------------------------
    # Rule 3 — Requests must increase (modulo)
    # ------------------------------------------
    for i in range(1, len(req_list)):
        prev = req_list[i-1]
        curr = req_list[i]
        diff = (curr - prev) % 256

        if diff == 0 or diff > max_step:
            incorrect.append(("REQ ORDER VIOLATION", req_seq_packets[i], prev, curr, diff))

    # ------------------------------------------
    # Rule 4 — Responses must increase (modulo)
    # ------------------------------------------
    for i in range(1, len(res_list)):
        prev = res_list[i-1]
        curr = res_list[i]
        diff = (curr - prev) % 256

        if diff == 0 or diff > max_step:
            incorrect.append(("RES ORDER VIOLATION", res_seq_packets[i], prev, curr, diff))

    # NOTE:
    # We DO NOT enforce global ordering rule anymore.
    # We DO NOT check cross-interaction (request → response).
    # Response SEQ may be lower than last request SEQ → ALLOWED.

    total = len(req_list) + len(res_list)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total else 0

    print("\n===== ZCL SEQUENCE ORDER REPORT — EXPERIMENT 2 =====")
    print(f"Total ZCL packets: {total}")
    print(f"Incorrect jumps: {error_count}")
    print(f"Sequence Compliance: {compliance:.2f}%")
    print("====================================================\n")

    if incorrect:
        print("---- Incorrect SEQ Packets ----")
        for err in incorrect[:20]:
            tag, pkt, prev, curr, diff = err
            print(f"{tag}: Prev={prev}, Curr={curr}, Jump={diff}")
            print(f"  Time: {pkt['time']}")
            print(f"  Info: {pkt['info']}\n")

    return compliance, incorrect


In [54]:
print("\n===== SUMMARY REPORT  - GPT 5 LOWW =====")
import pandas as pd
Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(10,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/New_Experiments/GPT5/GPT5_Exp2_Reasoning_low_Trial_{n}_10_minute_generated_message.json"

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    address_compliance, address_errors = check_address_compliance_exp2(valid_packets, VALID_SOURCES, VALID_DESTINATIONS)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    seq_compliance, seq_errors = check_seq_ordering_exp2(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    emr, matched_list = compute_exact_match_rate(valid_packets, Sample_Packets)



    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average EMR Rate: {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT  - GPT 5 LOWW =====

===== PACKET QUALITY REPORT =====
Total packets: 215
Valid packets: 215
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PROTOCOL COMPLIANCE REPORT =====
Total packets: 215
Incorrect protocol packets: 0
Protocol Compliance: 100.00%


===== PACKET QUALITY REPORT =====
Total packets: 215
Valid packets: 215
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== ADDRESS COMPLIANCE — EXPERIMENT 2 =====
Total packets: 215
Violations: 0
Compliance: 100.00%


===== PACKET QUALITY REPORT =====
Total packets: 215
Valid packets: 215
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== ZCL SEQUENCE ORDER REPORT — EXPERIMENT 2 =====
Total ZCL packets: 168
Incorrect jumps: 0
Sequence Compliance: 100.00%


===== PACKET QUALITY REPORT =====
Total packets: 215
Valid packets: 215
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PACKET REPETITION REPORT (NO 

In [55]:
print("\n===== SUMMARY REPORT  - GPT 5 MEDIUM =====")
import pandas as pd
Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(10,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/New_Experiments/GPT5/GPT5_Exp2_Reasoning_medium_Trial_{n}_10_minute_generated_message.json"

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    address_compliance, address_errors = check_address_compliance_exp2(valid_packets, VALID_SOURCES, VALID_DESTINATIONS)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    seq_compliance, seq_errors = check_seq_ordering_exp2(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    emr, matched_list = compute_exact_match_rate(valid_packets, Sample_Packets)



    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average EMR Rate: {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")



===== SUMMARY REPORT  - GPT 5 MEDIUM =====

===== PACKET QUALITY REPORT =====
Total packets: 214
Valid packets: 214
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PROTOCOL COMPLIANCE REPORT =====
Total packets: 214
Incorrect protocol packets: 0
Protocol Compliance: 100.00%


===== PACKET QUALITY REPORT =====
Total packets: 214
Valid packets: 214
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== ADDRESS COMPLIANCE — EXPERIMENT 2 =====
Total packets: 214
Violations: 0
Compliance: 100.00%


===== PACKET QUALITY REPORT =====
Total packets: 214
Valid packets: 214
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== ZCL SEQUENCE ORDER REPORT — EXPERIMENT 2 =====
Total ZCL packets: 168
Incorrect jumps: 0
Sequence Compliance: 100.00%


===== PACKET QUALITY REPORT =====
Total packets: 214
Valid packets: 214
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PACKET REPETITION REPORT (N

In [56]:
print("\n===== SUMMARY REPORT  - GPT 5 HIGH =====")
import pandas as pd
Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(10,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/New_Experiments/GPT5/GPT5_Exp2_Reasoning_high_Trial_{n}_10_minute_generated_message.json"

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    address_compliance, address_errors = check_address_compliance_exp2(valid_packets, VALID_SOURCES, VALID_DESTINATIONS)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    seq_compliance, seq_errors = check_seq_ordering_exp2(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)

    valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
    emr, matched_list = compute_exact_match_rate(valid_packets, Sample_Packets)



    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average EMR Rate: {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT  - GPT 5 HIGH =====

===== PACKET QUALITY REPORT =====
Total packets: 220
Valid packets: 220
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PROTOCOL COMPLIANCE REPORT =====
Total packets: 220
Incorrect protocol packets: 0
Protocol Compliance: 100.00%


===== PACKET QUALITY REPORT =====
Total packets: 220
Valid packets: 220
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== ADDRESS COMPLIANCE — EXPERIMENT 2 =====
Total packets: 220
Violations: 0
Compliance: 100.00%


===== PACKET QUALITY REPORT =====
Total packets: 220
Valid packets: 220
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== ZCL SEQUENCE ORDER REPORT — EXPERIMENT 2 =====
Total ZCL packets: 180
Incorrect jumps: 16
Sequence Compliance: 91.11%

---- Incorrect SEQ Packets ----
REQ ORDER VIOLATION: Prev=82, Curr=164, Jump=82
  Time: 52.143
  Info: ZCL: Read Attributes, Seq: 164

REQ ORDER VIOLATION: Prev=165, Curr=104, Jump

## EXPERIMENT 30 MINUTES


In [57]:
print("\n===== SUMMARY REPORT  - GPT 5 HIGH =====")
import pandas as pd
Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []


filepath = f"../Generated_Traffic/New_Experiments/Generalization_Ability/GPT5_Exp2_30_minute_max_token_generated_message.json"

valid_packets, corrupt_packets, decodability = analyze_llm_output(filepath)
protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
address_compliance, address_errors = check_address_compliance_exp2(valid_packets, VALID_SOURCES, VALID_DESTINATIONS)
seq_compliance, seq_errors = check_seq_ordering_exp2(valid_packets)
repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
emr, matched_list = compute_exact_match_rate(valid_packets, Sample_Packets)



Decodability.append(decodability)
Protocol_Compliance_Rate.append(protocol_compliance)
Address_Compliance_Rate.append(address_compliance)
Seq_Compliance_Rate.append(seq_compliance)
Repetition_Rate.append(repetition_rate)
Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average EMR Rate: {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT  - GPT 5 HIGH =====

===== PACKET QUALITY REPORT =====
Total packets: 598
Valid packets: 598
Corrupt packets: 0
Timestamp ordering issues: 0
Decodability: 100.00%


===== PROTOCOL COMPLIANCE REPORT =====
Total packets: 598
Incorrect protocol packets: 0
Protocol Compliance: 100.00%


===== ADDRESS COMPLIANCE — EXPERIMENT 2 =====
Total packets: 598
Violations: 0
Compliance: 100.00%


===== ZCL SEQUENCE ORDER REPORT — EXPERIMENT 2 =====
Total ZCL packets: 488
Incorrect jumps: 0
Sequence Compliance: 100.00%


===== PACKET REPETITION REPORT (NO LINK-STATUS) =====
Total NON-Link-Status packets: 488
Repeated packets: 0
Repetition rate: 0.00%

---- TOP 10 MOST REPEATED PATTERNS ----

===== EXACT MATCH RATE REPORT =====
Total generated packets: 598
Exact matches with real data: 1
Exact Match Rate: 0.17%

---- Example Exact-Matched Packets (first 10) ----
{'time': '32.400', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}

Avera